# 07 — Predictive Analysis: Falcon 9 Landing Classification
The predictive stage evaluates classification models on a held-out test set.

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

DATA_URL = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/dataset_part_3.csv"
data = pd.read_csv(DATA_URL)

X = data.drop(columns=["Class"])
y = data["Class"].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=2, stratify=y
)

print("Test sample size:", len(X_test))

In [ ]:
# SVM validation across the kernels used in the capstone.
svm_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("svc", SVC())
])

param_grid = {
    "svc__kernel": ["linear", "rbf", "sigmoid"],
    "svc__C": [0.5, 1, 10],
    "svc__gamma": ["scale", "auto"]
}

grid = GridSearchCV(svm_pipe, param_grid, cv=3, scoring="accuracy")
grid.fit(X_train, y_train)
print("Best SVM parameters:", grid.best_params_)
print("Best SVM validation accuracy:", grid.best_score_)

In [ ]:
# Tuned Decision Tree.
dt = DecisionTreeClassifier(random_state=2)
dt_grid = {
    "max_depth": [2, 4, 6, 8, 10],
    "min_samples_split": [2, 4, 6, 8, 10],
    "criterion": ["gini", "entropy"]
}

dt_search = GridSearchCV(dt, dt_grid, cv=3, scoring="accuracy")
dt_search.fit(X_train, y_train)

dt_pred = dt_search.predict(X_test)
dt_acc = accuracy_score(y_test, dt_pred)

print("Best Decision Tree parameters:", dt_search.best_params_)
print("Decision Tree test accuracy:", dt_acc)
print(confusion_matrix(y_test, dt_pred))
print(classification_report(y_test, dt_pred, zero_division=0))

### Verified course-task results

- Test sample size: **18 records**
- SVM validation result used in the course quiz: **linear kernel**
- Tuned Decision Tree test accuracy: **83.33%**

Because the held-out test sample is small, the 83.33% figure should be interpreted as a course evaluation result,
not as a production performance guarantee.